In [6]:
import os
import numpy as np
import tensorflow as tf
import librosa   # Only used at export time to compute identical mel filters


class PreprocessMel(tf.Module):
    def __init__(
        self,
        sr=4000,
        n_mels=128,
        n_fft=1024,
        hop_length=128,
        fmin=20,
        fmax=1500,
        amin=1e-10,
        top_db=80.0,
    ):
        super().__init__()
        self.sr = int(sr)
        self.n_mels = int(n_mels)
        self.n_fft = int(n_fft)
        self.hop_length = int(hop_length)
        self.fmin = float(fmin)
        self.fmax = float(fmax)
        self.amin = float(amin)
        self.top_db = float(top_db)

        # Precompute mel filterbank using librosa (ensures identical basis)
        mel_mat_np = librosa.filters.mel(
            sr=self.sr,
            n_fft=self.n_fft,
            n_mels=self.n_mels,
            fmin=self.fmin,
            fmax=self.fmax,
            htk=False,     # librosa default
            norm="slaney", # librosa default
        )
        # (F, M) where F = n_fft//2 + 1, M = n_mels
        self.mel_tf = tf.constant(mel_mat_np.T.astype(np.float32), dtype=tf.float32)

    @tf.function(input_signature=[tf.TensorSpec([1, None], dtype=tf.float32, name="wave")])
    def __call__(self, wave):
        """
        wave: [1, time]
        returns: [1, n_mels, time_frames]
        Equivalent to librosa.feature.melspectrogram -> power_to_db(..., ref=np.max, top_db=80)
        """
        wave = tf.squeeze(wave, axis=0)  # [time]

        # librosa.center=True, pad_mode='reflect'
        pad = tf.cast(self.n_fft // 2, tf.int32)
        wave = tf.pad(wave, [[pad, pad]], mode="REFLECT")

        # Hann-window STFT
        stft = tf.signal.stft(
            wave,
            frame_length=self.n_fft,
            frame_step=self.hop_length,
            fft_length=self.n_fft,
            window_fn=tf.signal.hann_window,
            pad_end=False,
        )  # [T, F_complex]

        # Power spectrogram (|STFT|^2)
        S = tf.square(tf.abs(stft))  # [T, F]

        # Apply mel filterbank → [T, M]
        mel_spec = tf.tensordot(S, self.mel_tf, axes=1)

        # librosa.power_to_db
        amin = tf.constant(self.amin, dtype=tf.float32)
        mel_spec = tf.maximum(mel_spec, amin)

        # Reference = global max
        ref_value = tf.reduce_max(mel_spec)
        log_spec = 10.0 * tf.math.log(mel_spec) / tf.math.log(10.0)
        log_spec = log_spec - (10.0 * tf.math.log(ref_value) / tf.math.log(10.0))

        # Top-dB clipping
        if self.top_db is not None:
            max_val = tf.reduce_max(log_spec)
            log_spec = tf.maximum(log_spec, max_val - self.top_db)

        # [T, M] → [M, T]
        log_mel_spec = tf.transpose(log_spec, perm=[1, 0])

        # Add batch dim → [1, M, T]
        return tf.expand_dims(log_mel_spec, axis=0)


if __name__ == "__main__":
    preprocess_model = PreprocessMel()

    saved_model_dir = "preprocess_mel_saved_model"
    tf.saved_model.save(preprocess_model, saved_model_dir)
    print(f"[INFO] Saved preprocessing model → {saved_model_dir}")

    # Convert to TFLite (STFT + mel filter requires SELECT_TF_OPS)
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    tflite_model = converter.convert()

    tflite_path = "preprocess_mel.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    print(f"[INFO] Converted TFLite model → {tflite_path}")


INFO:tensorflow:Assets written to: preprocess_mel_saved_model\assets


INFO:tensorflow:Assets written to: preprocess_mel_saved_model\assets


[INFO] Saved preprocessing model → preprocess_mel_saved_model
[INFO] Converted TFLite model → preprocess_mel.tflite


In [5]:
!python -c "import tensorflow as tf; print(tf.__version__)"

2.20.0


2025-09-15 10:38:16.967503: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-15 10:38:19.933141: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
import tensorflow as tf
print(tf.__version__)


2.20.0


In [ ]:
# Install the dependencies
%pip install numpy tensorflow librosa

  Using cached tensorflow-2.20.0-cp310-cp310-win_amd64.whl.metadata (4.6 kB)
Using cached tensorflow-2.20.0-cp310-cp310-win_amd64.whl (331.7 MB)


In [ ]:
%pip install --upgrade pip